In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from scipy.stats import zscore
from shapely.geometry import Polygon, Point

import geopandas as gpd
import h3

# reset working dir
import os
from pathlib import Path

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

In [3]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load weather data                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

weather = pd.read_csv("../data/weather/weather_hourly.csv")
print(weather.shape)
weather.head()

(17568, 15)


,date,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation
0,2024-01-01 00:00:00+00:00,-7.50,73.624560,-12.272507,0.0,0.0,0.0,0.0,891.08750,11.0,9.693296,13.797912,0.0,0.0,0.0
1,2024-01-01 01:00:00+00:00,-8.20,74.677930,-13.002723,0.0,0.0,0.0,0.0,890.57184,20.0,9.659814,12.429127,0.0,0.0,0.0
2,2024-01-01 02:00:00+00:00,-8.75,75.177630,-13.529382,0.0,0.0,0.0,0.0,889.95590,35.0,9.290511,9.693295,0.0,0.0,0.0
3,2024-01-01 03:00:00+00:00,-9.20,75.398346,-13.913002,0.0,0.0,0.0,0.0,889.04440,92.0,8.654987,6.489992,0.0,0.0,0.0
4,2024-01-01 04:00:00+00:00,-9.25,75.085120,-13.931759,0.0,0.0,0.0,0.0,888.67505,100.0,8.396570,3.415260,0.0,0.0,0.0


In [4]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 17568 entries, 0 to 17567
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  17568 non-null  str    
 1   temperature_2m        17568 non-null  float64
 2   relative_humidity_2m  17568 non-null  float64
 3   apparent_temperature  17568 non-null  float64
 4   precipitation         17568 non-null  float64
 5   rain                  17568 non-null  float64
 6   snowfall              17568 non-null  float64
 7   snow_depth            17568 non-null  float64
 8   surface_pressure      17568 non-null  float64
 9   cloud_cover           17568 non-null  float64
 10  wind_speed_10m        17568 non-null  float64
 11  wind_speed_100m       17568 non-null  float64
 12  is_day                17568 non-null  float64
 13  sunshine_duration     17568 non-null  float64
 14  direct_radiation      17568 non-null  float64
dtypes: float64(14), str(1)
memory 

In [5]:
display(weather)

,date,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation
0,2024-01-01 00:00:00+00:00,-7.50,73.624560,-12.272507,0.0,0.0,0.0,0.0,891.08750,11.0,9.693296,13.797912,0.0,0.0000,0.0
1,2024-01-01 01:00:00+00:00,-8.20,74.677930,-13.002723,0.0,0.0,0.0,0.0,890.57184,20.0,9.659814,12.429127,0.0,0.0000,0.0
2,2024-01-01 02:00:00+00:00,-8.75,75.177630,-13.529382,0.0,0.0,0.0,0.0,889.95590,35.0,9.290511,9.693295,0.0,0.0000,0.0
3,2024-01-01 03:00:00+00:00,-9.20,75.398346,-13.913002,0.0,0.0,0.0,0.0,889.04440,92.0,8.654987,6.489992,0.0,0.0000,0.0
4,2024-01-01 04:00:00+00:00,-9.25,75.085120,-13.931759,0.0,0.0,0.0,0.0,888.67505,100.0,8.396570,3.415260,0.0,0.0000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17563,2026-01-01 19:00:00+00:00,-4.85,66.939950,-8.355146,0.0,0.0,0.0,0.0,892.97750,0.0,1.698117,2.705993,0.0,1296.0045,7.0
17564,2026-01-01 20:00:00+00:00,-5.55,70.862970,-9.118733,0.0,0.0,0.0,0.0,892.38086,0.0,2.160000,0.763675,0.0,0.0000,0.0
17565,2026-01-01 21:00:00+00:00,-6.35,74.137620,-10.239040,0.0,0.0,0.0,0.0,892.33856,0.0,4.248152,0.254558,0.0,0.0000,0.0
17566,2026-01-01 22:00:00+00:00,-6.95,75.506676,-10.972478,0.0,0.0,0.0,0.0,892.21875,0.0,4.978554,1.800000,0.0,0.0000,0.0


weather timestamps are UTC-aware; convert to Chicago local time and strip tz:

In [6]:
weather["date"] = (
    pd.to_datetime(weather["date"], utc=True)
    .dt.tz_convert("America/Chicago")
    .dt.tz_localize(None)
)
weather = weather.sort_values("date")

Check for NaN values in weather:

In [7]:
weather.isna().sum().to_frame(name="Null Count").assign(
    Null_Percent=lambda x: (x["Null Count"] / len(weather) * 100).round(2)
)

,Null Count,Null_Percent
date,0,0.0
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
apparent_temperature,0,0.0
precipitation,0,0.0
rain,0,0.0
snowfall,0,0.0
snow_depth,0,0.0
surface_pressure,0,0.0
cloud_cover,0,0.0


check the time range:

In [8]:
print("Start Timestamp range:", weather["date"].min(), "to", weather["date"].max())

Start Timestamp range: 2023-12-31 18:00:00 to 2026-01-01 17:00:00


In [9]:
#limit the data to 2025 only

weather = weather[ (weather["date"]>= "2025-01-01") & (weather["date"] < "2026-01-01") ].reset_index()

print("Start Timestamp range:", weather["date"].min(), "to", weather["date"].max())

Start Timestamp range: 2025-01-01 00:00:00 to 2025-12-31 23:00:00


In [10]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   index                 8760 non-null   int64         
 1   date                  8760 non-null   datetime64[us]
 2   temperature_2m        8760 non-null   float64       
 3   relative_humidity_2m  8760 non-null   float64       
 4   apparent_temperature  8760 non-null   float64       
 5   precipitation         8760 non-null   float64       
 6   rain                  8760 non-null   float64       
 7   snowfall              8760 non-null   float64       
 8   snow_depth            8760 non-null   float64       
 9   surface_pressure      8760 non-null   float64       
 10  cloud_cover           8760 non-null   float64       
 11  wind_speed_10m        8760 non-null   float64       
 12  wind_speed_100m       8760 non-null   float64       
 13  is_day                8760 no

check for duplicates

In [11]:
print("Amount of duplicated rows: " + str(weather.duplicated().any().sum()))
print("Amount of duplicated dates: " + str(weather.date.duplicated().any().sum()))

Amount of duplicated rows: 0
Amount of duplicated dates: 1


In [12]:
weather[weather.date.duplicated()]

,index,date,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation
7321,16111,2025-11-02 01:00:00,4.0,62.128395,0.165052,0.0,0.0,0.0,0.0,888.17206,0.0,9.19939,6.310277,0.0,0.0,0.0


The duplicated date comes from the transition from daylight saving time, so its not erroneous

Check for gaps in the hourly data:

In [13]:
# create an hourly timerange as an attribute to compare with date
full_range = pd.date_range(
    start=weather.date.min(),
    end=weather.date.max(),
    freq="h"
)

In [14]:
missing_timestamps = full_range.difference(weather.date)
print("amount of gaps in 2025: " + str(len(missing_timestamps)))
missing_timestamps

amount of gaps in 2025: 1


DatetimeIndex(['2025-03-09 02:00:00'], dtype='datetime64[us]', freq='h')

In [15]:
weather.to_csv("../data/weather/STATE_01_04_CLEANED_WEATHER.csv", index=None)

Again, daylight saving time is the reason for the missing date.

In [16]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Outlier Detection                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# numeric_columns = ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth",
#                     "surface_pressure", "cloud_cover", "wind_speed_10m", "wind_speed_100m", "sunshine_duration", "direct_radiation"]

# # 2 plots per row
# rows = math.ceil(len(numeric_columns) / 2)

# fig, axes = plt.subplots(rows, 2, figsize=(12, rows * 4))
# axes = axes.flatten()

# for i, col in enumerate(numeric_columns):
#     data = pd.to_numeric(weather[col], errors='coerce').dropna()
#     axes[i].boxplot(data, vert=True)
#     axes[i].set_title(f'Boxplot of {col}')
#     axes[i].set_xlabel(col)
#     axes[i].grid(True)

# # Remove any unused axes
# for j in range(len(numeric_columns), len(axes)):
#     fig.delaxes(axes[j])

# plt.tight_layout()
# plt.show()


In [17]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Outlier Removal: Statistical Solution   #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# zscore_threshold = 3
# outlier_indices_per_column = {}
# all_outlier_indices = set()
# # remove outliers based on z-score
# for col in numeric_columns:
#     # Replace zeros with NaN and filter out non-positive values
#     safe_col = weather[col].replace(0, np.nan)
#     safe_col = safe_col[safe_col > 0]
    
#     if safe_col.empty:
#         continue
    
#     # Log-transform and compute z-score
#     log_transformed = np.log(safe_col)
#     zscores = np.abs(zscore(log_transformed))
#     zscore_series = pd.Series(zscores, index=log_transformed.index)

#     # Find outliers
#     outlier_indices = zscore_series[zscore_series > zscore_threshold].index
#     outlier_indices_per_column[col] = list(outlier_indices)
#     all_outlier_indices.update(outlier_indices)

#     print(f"--> {len(outlier_indices)} outliers detected in '{col}'")

# # Drop all outliers at once
# total_rows = len(weather) # for dropped percentage of dataset
# weather_outliers_removed = weather.drop(index=all_outlier_indices).reset_index(drop=True)

# removed_rows = len(all_outlier_indices)
# removed_percentage = round((removed_rows / total_rows) * 100, 2)

# print(f"\nOutlier detection completed. Total rows removed: {len(all_outlier_indices)}, {removed_percentage}%")

In [18]:
# # # # # # # # # # # # # # # # # # # # # # # # #
#                                               #
# Merge trip and weather_outliers_removed data  #
#                                               #
# # # # # # # # # # # # # # # # # # # # # # # # # 

# # check for duplicate timestamps and show them before removing
# n_dupes = weather_outliers_removed["date"].duplicated().sum()
# print(f"Duplicate weather_outliers_removed timestamps: {n_dupes}")
# if n_dupes > 0:
#     print(weather_outliers_removed[weather_outliers_removed["date"].duplicated(keep=False)].sort_values("date"))
#     weather_outliers_removed = weather_outliers_removed.drop_duplicates(subset="date")

# # create an hour-level key on the trip data
# raw_data["trip_start_hour"] = raw_data["trip_start_timestamp"].dt.floor("h")

# # merge on the hour key
# df_merged_outliers_removed = raw_data.merge(weather_outliers_removed, left_on="trip_start_hour", right_on="date", how="left")
# df_merged_outliers_removed = df_merged_outliers_removed.drop(columns=["date"])

# assert len(df_merged_outliers_removed) == len(raw_data), "Row count changed after merge — unexpected duplicates remain!"
# print(f"Merged shape: {df_merged_outliers_removed.shape}")
# print(f"Weather NAs after merge: {df_merged_outliers_removed['temperature_2m'].isna().sum()}")
# df_merged_outliers_removed.head()

In [19]:
# df_merged_outliers_removed.isna().sum().to_frame(name="Null Count").assign(
#     Null_Percent=lambda x: (x["Null Count"] / len(df_merged_outliers_removed) * 100).round(2)
# )

Removing the Rows containing missing values:

In [20]:
# df_merged_outliers_removed = df_merged_outliers_removed.dropna()

In [21]:
# df_merged_outliers_removed.to_csv("../data/merged/merged_outliers_removed.csv",index=None)